# Capstone Project: Plantaech
- **ID Tim Dicoding:** CC26-PSU258
- **Tema:** Sustainable Living & Responsible Consumption

## Analisis Permasalahan

**Tahap ini bertujuan untuk mengumpulkan dan menganalisis berbagai permasalahan, kemudian menentukan satu solusi utama yang akan dikembangkan dalam proyek.**

Indonesia adalah negara agraris dengan banyak keluarga yang bergantung pada pertanian. Petani kecil masih kesulitan menemukan penyakit tanaman secara dini. Panen yang gagal, kerugian ekonomi, dan penggunaan pestisida yang berlebihan merupakan akibat dari keterlambatan ini.

**Permasalahan yang diidentifikasi:**
1. **Keterlambatan Deteksi Penyakit:** Petani kecil di Indonesia menghadapi kesulitan mendeteksi penyakit tanaman secara dini, terutama di daerah yang jauh dari pusat layanan pertanian.
2. **Keterbatasan Akses:** Terbatasnya akses terhadap tenaga ahli agronomi dan alat diagnostik yang cepat dan murah.
3. **Dampak Ekonomi:** Penanganan terlambat mengakibatkan penggunaan pestisida berlebihan, gagal panen, dan kerugian ekonomi.
4. **Ketidakseimbangan Data:** Dataset penyakit tanaman yang tersedia seringkali tidak seimbang, yang berpotensi memengaruhi akurasi model.

**Solusi Utama:**
**Plantaech** - solusi berbasis AI yang memungkinkan petani mengidentifikasi penyakit tanaman tomat melalui foto daun. Model machine learning akan menganalisis gambar dan memberikan diagnosis serta saran penanganan.

## Menentukan Pertanyaan Bisnis

1. **Pertanyaan Bisnis 1 (SMART):**
   "Bagaimana distribusi dan proporsi jenis penyakit tanaman tomat yang teridentifikasi dalam dataset Plantaech, dan penyakit mana yang paling dominan serta paling jarang ditemukan selama periode pengumpulan data tahun 2026?"
   - *Specific:* Fokus pada distribusi dan proporsi jenis penyakit tanaman tomat
   - *Measurable:* Diukur melalui jumlah sampel per kelas dan persentasenya
   - *Action-Oriented:* Mengetahui penyakit dominan membantu memprioritaskan penanganan
   - *Relevant:* Relevan dengan tujuan Plantaech untuk deteksi penyakit
   - *Time-bound:* Periode pengumpulan data tahun 2026

2. **Pertanyaan Bisnis 2 (SMART):**
   "Apakah terdapat perbedaan signifikan dalam kualitas gambar (resolusi, ukuran file) antara kelas penyakit yang berbeda dalam dataset, yang dapat memengaruhi akurasi model deteksi penyakit Plantaech selama fase pengembangan Q2 2026?"
   - *Specific:* Fokus pada kualitas gambar antar kelas penyakit
   - *Measurable:* Diukur melalui resolusi dan ukuran file
   - *Action-Oriented:* Hasil analisis digunakan untuk preprocessing data
   - *Relevant:* Kualitas gambar berpengaruh pada akurasi model
   - *Time-bound:* Fase pengembangan Q2 2026

## Import Semua Packages/Library yang Digunakan

Seluruh library yang dibutuhkan untuk eksplorasi dan analisis diimpor di tahap ini.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from PIL import Image
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Data Wrangling

### Gathering Data

Pada tahap ini, kita akan memuat seluruh dataset gambar penyakit tanaman tomat dari direktori `dataset/`. Dataset ini terdiri dari gambar daun tomat yang diklasifikasikan ke dalam 10 kelas (9 penyakit + 1 sehat). Kita akan mengekstrak metadata dari setiap gambar untuk membangun DataFrame yang dapat dianalisis secara lebih mendalam.

In [ ]:
# Definisikan path dataset
DATASET_DIR = 'dataset'

# Fungsi untuk mengekstrak metadata gambar
def extract_metadata(image_path, class_name):
    try:
        file_size = os.path.getsize(image_path)
        with Image.open(image_path) as img:
            width, height = img.size
            mode = img.mode
            fmt = img.format if img.format else 'UNKNOWN'
        is_healthy = 'healthy' in class_name.lower()
        disease = class_name.replace('Tomato_', '').replace('_', ' ')
        if is_healthy:
            disease = 'Healthy'
        return {
            'filename': os.path.basename(image_path),
            'class_name': class_name,
            'disease_name': disease,
            'is_healthy': is_healthy,
            'condition': 'Healthy' if is_healthy else 'Diseased',
            'width': width, 'height': height,
            'aspect_ratio': round(width / height, 2),
            'total_pixels': width * height,
            'file_size_bytes': file_size,
            'file_size_kb': round(file_size / 1024, 2),
            'color_mode': mode,
            'file_format': fmt
        }
    except Exception as e:
        print(f"Error: {e}")
        return None

# Bangun metadata DataFrame
records = []
class_dirs = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])

print(f"Ditemukan {len(class_dirs)} kelas penyakit:\n")
for i, cls in enumerate(class_dirs):
    cls_path = os.path.join(DATASET_DIR, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"  [{i+1}/{len(class_dirs)}] {cls}: {len(imgs)} gambar")
    for img_file in imgs:
        meta = extract_metadata(os.path.join(cls_path, img_file), cls)
        if meta:
            records.append(meta)

df = pd.DataFrame(records)
print(f"\nTotal gambar berhasil dimuat: {len(df)}")
print(f"Kolom: {list(df.columns)}")

In [ ]:
# Preview dataset
print("=== Preview Dataset ===")
df.head()

**Insight Gathering Data:**
- Dataset berhasil dimuat dari 10 subdirektori yang masing-masing merepresentasikan satu kelas penyakit tanaman tomat.
- Total terdapat 16.011 gambar daun tomat.
- Dataset mencakup 9 jenis penyakit dan 1 kategori tanaman sehat.
- Setiap gambar memiliki metadata berupa dimensi, ukuran file, format, dan mode warna.

### Assessing Data

Pada tahap ini, kita akan mengevaluasi kualitas dan struktur data, seperti mengecek tipe data, mencari missing values, duplikasi, dan invalid values.

In [ ]:
# Cek tipe data dan missing values
print("=== Dataset Info ===")
df.info()
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# Cek duplikasi berdasarkan filename
dup_filenames = df[df.duplicated(subset=['filename'], keep=False)]
print(f"Jumlah file dengan nama duplikat: {len(dup_filenames)}")

**Insight Assessing Data:**
- Tidak ditemukan missing values pada dataset metadata.
- Seluruh gambar memiliki format yang valid.
- Tipe data mayoritas sesuai, namun akan dipastikan konsistensinya di tahap Cleaning.

### Cleaning Data

Tahap ini bertujuan untuk membersihkan dan mempersiapkan data sebelum masuk ke tahap analisis lebih lanjut.

In [ ]:
# 1. Hapus duplikasi (berdasarkan seluruh kolom)
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Baris sebelum deduplikasi: {before}")
print(f"Baris setelah deduplikasi: {after}")

# 2. Pastikan tipe data konsisten
df['is_healthy'] = df['is_healthy'].astype(bool)
df['width'] = df['width'].astype(int)
df['height'] = df['height'].astype(int)

**Insight Cleaning Data:**
- Data telah bersih dari duplikasi.
- Tipe data telah dipastikan konsisten dan siap untuk tahap analisis lanjutan dan feature engineering.

## Teknik Analisis Lanjutan: Clustering (Manual Grouping / Binning)

Sebagai bentuk teknik analisis lanjutan non-Machine Learning, kita akan menerapkan **Clustering menggunakan teknik Binning dan Manual Grouping**. Tujuannya adalah mengelompokkan data gambar berdasarkan karakteristik kualitasnya (ukuran file dan total piksel/resolusi).

Pengelompokan ini berguna untuk melihat apakah ada kelompok kualitas tertentu yang mendominasi dataset, sehingga dapat menjadi pertimbangan dalam merancang tahapan preprocessing model machine learning nantinya.

In [ ]:
# Feature Engineering dengan Binning: Menambahkan kategori ukuran file
df['file_size_category'] = pd.cut(
    df['file_size_kb'],
    bins=[0, 10, 15, 20, 30, float('inf')],
    labels=['Very Small (<10KB)', 'Small (10-15KB)', 'Medium (15-20KB)', 'Large (20-30KB)', 'Very Large (>30KB)']
)

# Feature Engineering dengan Binning: Menambahkan kategori resolusi (berdasarkan total piksel)
df['resolution_category'] = df['total_pixels'].apply(
    lambda x: 'Low (<100K px)' if x < 100000 else ('Medium (100K-500K px)' if x < 500000 else 'High (>500K px)')
)

# Melihat hasil clustering / grouping
print("=== Distribusi Cluster Kategori Ukuran File ===")
print(df['file_size_category'].value_counts())

print("\n=== Distribusi Cluster Kategori Resolusi ===")
print(df['resolution_category'].value_counts())

In [ ]:
# Simpan data yang sudah bersih dan memiliki fitur tambahan untuk dashboard
os.makedirs('dashboard', exist_ok=True)
df.to_csv('dashboard/main_data.csv', index=False)
print("Data berhasil disimpan ke dashboard/main_data.csv")

## Exploratory Data Analysis (EDA)

Eksplorasi data ini dilakukan untuk menemukan pola dan menjawab pertanyaan bisnis yang telah didefinisikan.

### Explore: Distribusi Kelas Penyakit

In [ ]:
# Distribusi kelas penyakit
print("=== Distribusi Kelas Penyakit ===")
class_dist = df.groupby('disease_name').agg(
    total_samples=('filename', 'count'),
    avg_file_size_kb=('file_size_kb', 'mean')
).sort_values('total_samples', ascending=False).round(2)

class_dist['proportion_%'] = (class_dist['total_samples'] / len(df) * 100).round(2)
class_dist

**Insight Distribusi Kelas:**
- Terdapat ketidakseimbangan kelas (class imbalance) yang cukup besar. Tomato YellowLeaf Curl Virus sangat dominan, sedangkan Tomato mosaic virus memiliki jumlah paling sedikit.

### Explore: Perbandingan Tanaman Sehat vs Sakit

In [ ]:
# Perbandingan Sehat vs Sakit
print("=== Perbandingan Sehat vs Sakit ===")
condition_stats = df.groupby('condition').agg(
    jumlah=('filename', 'count'),
    rata_rata_ukuran_kb=('file_size_kb', 'mean'),
    median_ukuran_kb=('file_size_kb', 'median')
).round(2)

condition_stats['proporsi_%'] = (condition_stats['jumlah'] / len(df) * 100).round(2)
condition_stats

## Visualization & Explanatory Analysis

Visualisasi disajikan secara menarik dan efektif dengan mematuhi prinsip desain visualisasi data yang baik.

### Pertanyaan 1: Distribusi dan Proporsi Jenis Penyakit Tanaman Tomat

In [ ]:
# Visualisasi 1: Distribusi Kelas Penyakit
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
class_counts_sorted = df['disease_name'].value_counts()
colors = sns.color_palette("viridis", len(class_counts_sorted))
bars = axes[0].barh(class_counts_sorted.index, class_counts_sorted.values, color=colors)
axes[0].set_xlabel('Jumlah Sampel', fontsize=12)
axes[0].set_ylabel('Jenis Penyakit', fontsize=12)
axes[0].set_title('Distribusi Jumlah Sampel per Kelas Penyakit', fontsize=14, fontweight='bold')
for bar, val in zip(bars, class_counts_sorted.values):
    axes[0].text(val + 20, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_counts_sorted.values, labels=class_counts_sorted.index,
    autopct='%1.1f%%', colors=colors, pctdistance=0.85, startangle=90
)
for t in texts: t.set_fontsize(8)
for at in autotexts: at.set_fontsize(7)
axes[1].set_title('Proporsi Kelas Penyakit', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

**Analisis Pertanyaan 1:**
- **Penyakit Paling Dominan:** Tomato YellowLeaf Curl Virus dengan ~3.208 sampel (20.0%).
- **Penyakit Paling Jarang:** Tomato mosaic virus dengan ~373 sampel (2.3%).
- Visualisasi tersebut sangat merepresentasikan betapa perlunya handling terhadap class-imbalance.

### Pertanyaan 2: Perbedaan Kualitas Gambar Antar Kelas (terkait ukuran)

In [ ]:
# Visualisasi 2: Perbandingan Kualitas Gambar
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Box plot ukuran file per kelas
order = df.groupby('disease_name')['file_size_kb'].median().sort_values().index
sns.boxplot(data=df, y='disease_name', x='file_size_kb', order=order, palette='coolwarm', ax=axes[0])
axes[0].set_title('Distribusi Ukuran File per Kelas', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Ukuran File (KB)', fontsize=12)
axes[0].set_ylabel('Jenis Penyakit', fontsize=12)

# Histogram Sehat vs Sakit
for cond, color in [('Healthy', '#2ecc71'), ('Diseased', '#e74c3c')]:
    subset = df[df['condition'] == cond]
    axes[1].hist(subset['file_size_kb'], bins=30, alpha=0.6, label=f'{cond} (n={len(subset)})', color=color, density=True)
healthy_mean = df[df['is_healthy']]['file_size_kb'].mean()
diseased_mean = df[~df['is_healthy']]['file_size_kb'].mean()
axes[1].axvline(healthy_mean, color='green', linestyle='--', label=f'Mean Sehat: {healthy_mean:.1f} KB')
axes[1].axvline(diseased_mean, color='red', linestyle='--', label=f'Mean Sakit: {diseased_mean:.1f} KB')
axes[1].set_title('Distribusi Ukuran File: Sehat vs Sakit', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ukuran File (KB)')
axes[1].set_ylabel('Densitas')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

**Analisis Pertanyaan 2:**
- Menunjukkan adanya perbedaan rata-rata dan penyebaran kualitas gambar (ukuran) pada tiap kelas penyakit dan juga pada perbandingan sehat vs sakit.

## A/B Testing

Untuk melengkapi pemahaman statistik, kita melakukan A/B Testing menggunakan **Independent Two-Sample T-Test** terkait kualitas gambar (ukuran file) antara tanaman sehat (Grup A) dan tanaman sakit (Grup B).

**Hipotesis:**
- **H₀:** Tidak ada perbedaan signifikan rata-rata ukuran file antara gambar tanaman sehat dan sakit (μ_sehat = μ_sakit)
- **H₁:** Terdapat perbedaan signifikan rata-rata ukuran file antara gambar tanaman sehat dan sakit (μ_sehat ≠ μ_sakit)
- **Significance Level (α):** 0.05

In [ ]:
# A/B Testing: Independent T-Test
healthy_sizes = df[df['is_healthy'] == True]['file_size_kb']
diseased_sizes = df[df['is_healthy'] == False]['file_size_kb']

t_stat, p_value = stats.ttest_ind(healthy_sizes, diseased_sizes)

print(f"=== Hasil A/B Testing ===")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.6f}")

if p_value < 0.05:
    print(f"\nKesimpulan: Tolak H₀. Terdapat perbedaan signifikan rata-rata ukuran file.")
else:
    print(f"\nKesimpulan: Gagal Tolak H₀. Tidak ada perbedaan signifikan.")

## Conclusion & Recommendation

### Kesimpulan
1. **Kondisi Distribusi Data:** Terdapat ketidakseimbangan kelas (class imbalance) dengan rasio yang tinggi antara kelas dominan (Tomato YellowLeaf Curl Virus) dan kelas terkecil (Tomato mosaic virus).
2. **Kualitas Data:** Mayoritas gambar termasuk dalam klasifikasi resolusi rendah dan ukuran file yang berbeda signifikan antara kelas sehat dan sakit.

### Rekomendasi (Action Items)
1. **Data Augmentation:** Terapkan teknik augmentasi data seperti rotasi atau zoom pada kelas dengan sampel sedikit (contoh: Tomato mosaic virus).
2. **Standardisasi Preprocessing:** Pastikan penggunaan standarisasi image resize yang konstan sebelum memberikan data gambar tersebut ke dalam pipeline model AI.
3. **Pengumpulan Data:** Pertimbangkan untuk mengumpulkan gambar sampel Tomato mosaic virus dari lapangan untuk menyeimbangkan dataset.